# Structured Data Exploitation Zone

This notebook transforms the cleaned Trusted Zone tables in ClickHouse into exploitation-ready dimensional models and analytical marts.

The design follows two rules:

- `natural_disaster_tweets`: text-oriented feature extraction.
- `global_warming_dataset`, `temperature_change`, and `co2_emission_by_vehicles`: dimensional modelling plus denormalized analytical marts.

## Purpose
Model trusted structured tables into exploitation dimensions, facts, bridges, marts, and governance evidence for analytics and ML.

## Inputs
Trusted ClickHouse tables in `bi_analytics`: global warming, temperature change, vehicle emissions, and natural disaster tweets.

## Validation / quality checks
The notebook verifies trusted table availability, non-empty outputs, metadata completeness, dimension key uniqueness, and exploitation quality summary records.

## Transformation / cleaning logic
SQL transformations create reusable dimensions, analytical facts, tweet text features, bridge tables, and denormalized marts without changing trusted source semantics.

## Metadata and lineage fields
Exploitation outputs include `source_system`, `source_assets`, `created_at`, `schema_version`, `exploitation_asset_type`, catalogue rows, quality checks, and lineage records.

## Output assets
ClickHouse assets under `exploitation_analytics`, including `dim_*`, `fact_*`, `bridge_*`, `mart_*`, `exploitation_catalogue`, `exploitation_quality_summary`, and `exploitation_lineage`.

## RBAC / service user used
The production DAG uses ClickHouse exploitation writer/reader roles while reading trusted tables with approved service credentials.

## Notebook-DAG alignment note
This notebook mirrors `exploitation_zone_structured`; it does not import DAG functions, and the SQL/table names remain aligned with the DAG implementation.


## Execution Notes and Batch Logging
This notebook builds structured exploitation products in ClickHouse: trusted-table validation, dimensions, facts, marts, and governance evidence. Batch-style prints identify each source validation, dimension/table creation, preview, and governance scan so the run output reads like an execution trace.


## 1. Environment Setup

### Setup and Constants


In [1]:
# Import the ClickHouse client used to create exploitation-zone tables.
from string import Formatter
from datetime import datetime, timezone

import clickhouse_connect
import pandas as pd

# Source database: cleaned and standardized Trusted Zone tables.
TRUSTED_DB = "bi_analytics"

# Target database: curated tables for analytics, dashboards, and ML features.
EXPLOITATION_DB = "exploitation_analytics"

# Notebook-local metadata constants mirroring the structured exploitation DAG.
EXPLOITATION_CREATED_AT = datetime.now(timezone.utc).isoformat()
EXPLOITATION_SCHEMA_VERSION = "exploitation_structured_v1"

# Logical source table names used throughout the exploitation model.
GW_TABLE = "global_warming_dataset"
TEMP_TABLE = "temperature_change"
VEHICLE_TABLE = "co2_emission_by_vehicles"
TWEET_TABLE = "natural_disaster_tweets"

TRUSTED_TABLES = [TWEET_TABLE, GW_TABLE, TEMP_TABLE, VEHICLE_TABLE]

# Candidate lists keep exploitation resilient to small trusted-zone naming improvements.
# The first item is the preferred canonical lowercase name; later items are accepted legacy forms.
COLUMN_CANDIDATES = {
    GW_TABLE: {},
    TEMP_TABLE: {},
    TWEET_TABLE: {},
    VEHICLE_TABLE: {
        "engine_size_l": ["engine_size_l", "engine_sizel"],
        "co2_emissions_g_km": ["co2_emissions_g_km", "co2_emissionsg_km"],
    },
}

# Create one reusable ClickHouse connection for the whole notebook.
client = clickhouse_connect.get_client(
    host="clickhouse",
    port=8123,
    username="analytics",
    password="analytics_secret",
)



### Source Resolution Helpers


In [2]:

def quote_identifier(identifier: str) -> str:
    """Quote a ClickHouse identifier without changing its case."""
    return "`" + identifier.replace("`", "``") + "`"


def table_ref(database_name: str, table_name: str) -> str:
    """Build a fully-qualified ClickHouse table reference."""
    return f"{quote_identifier(database_name)}.{quote_identifier(table_name)}"


def trusted_ref(table_name: str) -> str:
    """Return a fully-qualified Trusted Zone table reference."""
    return table_ref(TRUSTED_DB, table_name)


def exploitation_ref(table_name: str) -> str:
    """Return a fully-qualified Exploitation Zone table reference."""
    return table_ref(EXPLOITATION_DB, table_name)


_TRUSTED_COLUMN_CACHE: dict[str, set[str]] = {}


def trusted_columns(table_name: str) -> set[str]:
    """Read the actual Trusted Zone schema from ClickHouse once per table."""
    if table_name not in _TRUSTED_COLUMN_CACHE:
        rows = client.query(
            f"""
            SELECT name
            FROM system.columns
            WHERE database = '{TRUSTED_DB}'
              AND table = '{table_name}'
            """
        ).result_rows
        _TRUSTED_COLUMN_CACHE[table_name] = {row[0] for row in rows}
    return _TRUSTED_COLUMN_CACHE[table_name]


def resolve_column(table_name: str, logical_name: str) -> str:
    """Resolve a logical business field to the physical Trusted Zone column name."""
    columns = trusted_columns(table_name)
    candidates = COLUMN_CANDIDATES.get(table_name, {}).get(logical_name, [logical_name])

    for candidate in candidates:
        if candidate in columns:
            return candidate

    raise KeyError(
        f"Cannot resolve column '{logical_name}' in {trusted_ref(table_name)}. "
        f"Tried {candidates}; available columns: {sorted(columns)}"
    )


def source_col(table_name: str, logical_name: str) -> str:
    """Return a quoted physical source column for generated SQL."""
    return quote_identifier(resolve_column(table_name, logical_name))




### Create Exploitation Database


In [3]:
# Create the exploitation database if this notebook is executed for the first time.
client.command(f"CREATE DATABASE IF NOT EXISTS {quote_identifier(EXPLOITATION_DB)}")
print(f"Connected to ClickHouse. Target database: {EXPLOITATION_DB}")


Connected to ClickHouse. Target database: exploitation_analytics


## 2. Trusted Zone Validation

In [4]:
# Validate that every expected trusted table exists and contains rows before modelling.
for batch_no, table_name in enumerate(TRUSTED_TABLES, start=1):
    row_count = client.query(f"SELECT count() FROM {trusted_ref(table_name)}").first_row[0]
    columns = trusted_columns(table_name)
    print(f"[trusted validation batch {batch_no}/{len(TRUSTED_TABLES)}] {table_name:<32} {row_count:>10,} rows | {len(columns):>3} columns")


[trusted validation batch 1/4] natural_disaster_tweets             107,593 rows |  10 columns
[trusted validation batch 2/4] global_warming_dataset              100,000 rows |  31 columns
[trusted validation batch 3/4] temperature_change                  241,893 rows |  19 columns
[trusted validation batch 4/4] co2_emission_by_vehicles              5,988 rows |  17 columns


## 3. Helper Functions

In [5]:
# This helper makes every table creation idempotent: rerunning the notebook replaces old outputs.


def exploitation_asset_type(table_name: str) -> str:
    if table_name.startswith("dim_"):
        return "dimension"
    if table_name.startswith("fact_"):
        return "fact"
    if table_name.startswith("bridge_"):
        return "bridge"
    if table_name.startswith("mart_"):
        return "mart"
    return "metric"

def sql_literal(value: str) -> str:
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"

def recreate_table(table_name: str, select_sql: str, order_by: str, source_assets: str | None = None) -> None:
    """Create an exploitation table from a SELECT statement in an idempotent way."""
    full_name = exploitation_ref(table_name)
    source_assets = source_assets or ",".join(TRUSTED_TABLES)
    metadata_sql = ",\n        ".join([
        f"{sql_literal('trusted-zone')} AS `source_system`",
        f"{sql_literal(source_assets)} AS `source_assets`",
        f"{sql_literal(EXPLOITATION_CREATED_AT)} AS `created_at`",
        f"{sql_literal(EXPLOITATION_SCHEMA_VERSION)} AS `schema_version`",
        f"{sql_literal(exploitation_asset_type(table_name))} AS `exploitation_asset_type`",
    ])
    print(f"[table build] Creating {full_name} from sources={source_assets}")
    client.command(f"DROP TABLE IF EXISTS {full_name}")
    client.command(
        f"""
        CREATE TABLE {full_name}
        ENGINE = MergeTree()
        ORDER BY {order_by}
        SETTINGS allow_nullable_key = 1
        AS
        SELECT
            __base.*,
            {metadata_sql}
        FROM (
        {select_sql}
        ) AS __base
        """
    )
    row_count = client.query(f"SELECT count() FROM {full_name}").first_row[0]
    print(f"Created {full_name:<55} {row_count:>10,} rows")


def template_fields(template: str) -> set[str]:
    """Return placeholder names used by a SQL expression template."""
    return {field_name for _, field_name, _, _ in Formatter().parse(template) if field_name}


def render_source_template(table_name: str, template: str, column_map: dict[str, str]) -> str:
    """Replace logical placeholders with quoted Trusted Zone source columns."""
    resolved_columns = {
        placeholder: source_col(table_name, column_map.get(placeholder, placeholder))
        for placeholder in template_fields(template)
    }
    return template.format_map(resolved_columns)


def build_dimension_select(definition: dict, source_config: dict) -> str:
    """Build one SELECT DISTINCT branch for a reusable dimension definition."""
    table_name = source_config["table"]
    column_map = source_config.get("columns", {})
    fields = source_config.get("fields", definition["fields"])

    select_list = ",\n        ".join(
        f"{render_source_template(table_name, expression, column_map)} AS {quote_identifier(output_name)}"
        for output_name, expression in fields.items()
    )
    where_template = source_config.get("where", definition.get("where"))
    where_clause = ""
    if where_template:
        where_clause = f"\n    WHERE {render_source_template(table_name, where_template, column_map)}"

    return f"""
    SELECT DISTINCT
        {select_list}
    FROM {trusted_ref(table_name)}{where_clause}
    """.strip()


def recreate_dimension_table(table_name: str, definition: dict) -> None:
    """Create a dimension table by unioning configured distinct values from one or more sources."""
    select_sql = "\n    UNION DISTINCT\n".join(
        build_dimension_select(definition, source_config)
        for source_config in definition["sources"]
    )
    recreate_table(table_name, select_sql, definition["order_by"])


## 4. Dimension Tables

Dimension tables describe the main analytical entities used by the fact tables and marts. They make the model easier to query because repeated descriptive values are centralized into reusable lookup tables.

| Table | Grain | Purpose |
|---|---|---|
| `dim_year` | One row per year | Provides a shared time dimension and adds `decade` for long-term trend analysis. |
| `dim_country` | One row per country from the climate dataset | Connects country-level climate facts to readable country names. |
| `dim_area` | One row per temperature-change area | Describes geographic areas in the temperature dataset, including the M49 area code. |
| `dim_time_period` | One row per month, season, or annual period label/code | Supports monthly, seasonal, and annual temperature trend analysis. |
| `dim_disaster_type` | One row per disaster category | Standardizes tweet disaster categories such as flood, earthquake, or wildfire. |
| `dim_vehicle` | One row per vehicle configuration | Describes make, model, class, engine, transmission, and fuel attributes for vehicle-emission analysis. |

In [6]:
# Build reusable dimensions from compact definitions.
# Add a new source to a dimension by appending one item under "sources" instead of writing another SELECT block.
DIMENSION_DEFINITIONS = {
    "dim_year": {
        "order_by": "year_key",
        "fields": {
            "year_key": "toInt32({year})",
            "year": "toInt32({year})",
            "decade": "intDiv(toInt32({year}), 10) * 10",
        },
        "sources": [
            {"table": GW_TABLE},
            {"table": TEMP_TABLE},
        ],
    },
    "dim_country": {
        "order_by": "country_key",
        "fields": {
            "country_key": "cityHash64({country_name})",
            "country_name": "{country_name}",
        },
        "sources": [
            {"table": GW_TABLE, "columns": {"country_name": "country"}, "where": "{country_name} != ''"},
        ],
    },
    "dim_area": {
        "order_by": "area_key",
        "fields": {
            "area_key": "cityHash64({area_name})",
            "area_code_m49": "{area_code_m49}",
            "area_name": "{area_name}",
        },
        "sources": [
            {"table": TEMP_TABLE, "columns": {"area_name": "area"}, "where": "{area_name} != ''"},
        ],
    },
    "dim_time_period": {
        "order_by": "period_key",
        "fields": {
            "period_key": "{period_key}",
            "period_name": "{period_name}",
            "period_type": "multiIf(position(ifNull({period_name}, ''), '-') > 0, 'season', positionCaseInsensitive(ifNull({period_name}, ''), 'year') > 0, 'annual', 'month')",
        },
        "sources": [
            {"table": TEMP_TABLE, "columns": {"period_key": "months_code", "period_name": "months"}, "where": "{period_name} != ''"},
        ],
    },
    "dim_disaster_type": {
        "order_by": "disaster_type_key",
        "fields": {
            "disaster_type_key": "cityHash64({disaster_type})",
            "disaster_type": "{disaster_type}",
        },
        "sources": [
            {"table": TWEET_TABLE, "where": "{disaster_type} != ''"},
        ],
    },
    "dim_vehicle": {
        "order_by": "vehicle_key",
        "fields": {
            "vehicle_key": "cityHash64({make}, {model}, {vehicle_class}, {transmission}, {fuel_type}, {engine_size_l}, {cylinders})",
            "make": "{make}",
            "model": "{model}",
            "vehicle_class": "{vehicle_class}",
            "engine_size_l": "{engine_size_l}",
            "cylinders": "{cylinders}",
            "transmission": "{transmission}",
            "fuel_type": "{fuel_type}",
        },
        "sources": [
            {"table": VEHICLE_TABLE},
        ],
    },
}

for batch_no, (dimension_name, definition) in enumerate(DIMENSION_DEFINITIONS.items(), start=1):
    print(f"[dimension batch {batch_no}/{len(DIMENSION_DEFINITIONS)}] {dimension_name}")
    recreate_dimension_table(dimension_name, definition)


[dimension batch 1/6] dim_year
[table build] Creating `exploitation_analytics`.`dim_year` from sources=natural_disaster_tweets,global_warming_dataset,temperature_change,co2_emission_by_vehicles
Created `exploitation_analytics`.`dim_year`                            124 rows
[dimension batch 2/6] dim_country
[table build] Creating `exploitation_analytics`.`dim_country` from sources=natural_disaster_tweets,global_warming_dataset,temperature_change,co2_emission_by_vehicles
Created `exploitation_analytics`.`dim_country`                         195 rows
[dimension batch 3/6] dim_area
[table build] Creating `exploitation_analytics`.`dim_area` from sources=natural_disaster_tweets,global_warming_dataset,temperature_change,co2_emission_by_vehicles
Created `exploitation_analytics`.`dim_area`                            247 rows
[dimension batch 4/6] dim_time_period
[table build] Creating `exploitation_analytics`.`dim_time_period` from sources=natural_disaster_tweets,global_warming_dataset,temperat

## 5. Fact Tables

Fact tables store measurable events or observations at a clear grain. They keep the source data analytically structured while replacing repeated descriptive columns with dimension keys.

| Table | Grain | Purpose |
|---|---|---|
| `fact_climate_country_year` | One country per year | Stores climate, economy, population, emissions, policy, and environmental indicators for country-year analysis. |
| `fact_temperature_area_period` | One area per time period per year | Stores monthly, seasonal, and annual temperature-change values and quality flags for time-series trend analysis. |
| `fact_vehicle_emission` | One vehicle-emission record | Stores fuel-consumption and CO2-emission measurements linked to vehicle attributes. |

In [7]:
# Build fact tables at the natural analytical grain of each trusted structured dataset.
# These tables are normalized enough for reuse, while marts later denormalize them for BI consumption.
recreate_table(
    "fact_climate_country_year",
    f"""
    SELECT
        cityHash64({source_col(GW_TABLE, 'country')}) AS country_key,
        toInt32({source_col(GW_TABLE, 'year')}) AS year_key,
        {source_col(GW_TABLE, 'temperature_anomaly')} AS temperature_anomaly,
        {source_col(GW_TABLE, 'average_temperature')} AS average_temperature,
        {source_col(GW_TABLE, 'co2_emissions')} AS co2_emissions,
        {source_col(GW_TABLE, 'population')} AS population,
        {source_col(GW_TABLE, 'forest_area')} AS forest_area,
        {source_col(GW_TABLE, 'gdp')} AS gdp,
        {source_col(GW_TABLE, 'renewable_energy_usage')} AS renewable_energy_usage,
        {source_col(GW_TABLE, 'methane_emissions')} AS methane_emissions,
        {source_col(GW_TABLE, 'sea_level_rise')} AS sea_level_rise,
        {source_col(GW_TABLE, 'arctic_ice_extent')} AS arctic_ice_extent,
        {source_col(GW_TABLE, 'urbanization')} AS urbanization,
        {source_col(GW_TABLE, 'deforestation_rate')} AS deforestation_rate,
        {source_col(GW_TABLE, 'extreme_weather_events')} AS extreme_weather_events,
        {source_col(GW_TABLE, 'average_rainfall')} AS average_rainfall,
        {source_col(GW_TABLE, 'solar_energy_potential')} AS solar_energy_potential,
        {source_col(GW_TABLE, 'waste_management')} AS waste_management,
        {source_col(GW_TABLE, 'per_capita_emissions')} AS source_per_capita_emissions,
        {source_col(GW_TABLE, 'industrial_activity')} AS industrial_activity,
        {source_col(GW_TABLE, 'air_pollution_index')} AS air_pollution_index,
        {source_col(GW_TABLE, 'biodiversity_index')} AS biodiversity_index,
        {source_col(GW_TABLE, 'ocean_acidification')} AS ocean_acidification,
        {source_col(GW_TABLE, 'fossil_fuel_usage')} AS fossil_fuel_usage,
        {source_col(GW_TABLE, 'energy_consumption_per_capita')} AS energy_consumption_per_capita,
        {source_col(GW_TABLE, 'policy_score')} AS policy_score
    FROM {trusted_ref(GW_TABLE)}
    """,
    "(country_key, year_key)",
)

recreate_table(
    "fact_temperature_area_period",
    f"""
    SELECT
        cityHash64({source_col(TEMP_TABLE, 'area')}) AS area_key,
        toInt32({source_col(TEMP_TABLE, 'year')}) AS year_key,
        {source_col(TEMP_TABLE, 'months_code')} AS period_key,
        {source_col(TEMP_TABLE, 'domain_code')} AS domain_code,
        {source_col(TEMP_TABLE, 'domain')} AS domain,
        {source_col(TEMP_TABLE, 'element_code')} AS element_code,
        {source_col(TEMP_TABLE, 'element')} AS element,
        {source_col(TEMP_TABLE, 'year_code')} AS year_code,
        {source_col(TEMP_TABLE, 'unit')} AS unit,
        {source_col(TEMP_TABLE, 'value')} AS temperature_change_value,
        {source_col(TEMP_TABLE, 'flag')} AS flag,
        {source_col(TEMP_TABLE, 'flag_description')} AS flag_description
    FROM {trusted_ref(TEMP_TABLE)}
    """,
    "(area_key, year_key, period_key)",
)

recreate_table(
    "fact_vehicle_emission",
    f"""
    SELECT
        cityHash64(
            {source_col(VEHICLE_TABLE, 'make')},
            {source_col(VEHICLE_TABLE, 'model')},
            {source_col(VEHICLE_TABLE, 'vehicle_class')},
            {source_col(VEHICLE_TABLE, 'transmission')},
            {source_col(VEHICLE_TABLE, 'fuel_type')},
            {source_col(VEHICLE_TABLE, 'engine_size_l')},
            {source_col(VEHICLE_TABLE, 'cylinders')}
        ) AS vehicle_key,
        {source_col(VEHICLE_TABLE, 'fuel_consumption_city_l_100_km')} AS fuel_consumption_city_l_100_km,
        {source_col(VEHICLE_TABLE, 'fuel_consumption_hwy_l_100_km')} AS fuel_consumption_hwy_l_100_km,
        {source_col(VEHICLE_TABLE, 'fuel_consumption_comb_l_100_km')} AS fuel_consumption_comb_l_100_km,
        {source_col(VEHICLE_TABLE, 'fuel_consumption_comb_mpg')} AS fuel_consumption_comb_mpg,
        {source_col(VEHICLE_TABLE, 'co2_emissions_g_km')} AS co2_emissions_g_km
    FROM {trusted_ref(VEHICLE_TABLE)}
    """,
    "vehicle_key",
)


[table build] Creating `exploitation_analytics`.`fact_climate_country_year` from sources=natural_disaster_tweets,global_warming_dataset,temperature_change,co2_emission_by_vehicles
Created `exploitation_analytics`.`fact_climate_country_year`       100,000 rows
[table build] Creating `exploitation_analytics`.`fact_temperature_area_period` from sources=natural_disaster_tweets,global_warming_dataset,temperature_change,co2_emission_by_vehicles
Created `exploitation_analytics`.`fact_temperature_area_period`    241,893 rows
[table build] Creating `exploitation_analytics`.`fact_vehicle_emission` from sources=natural_disaster_tweets,global_warming_dataset,temperature_change,co2_emission_by_vehicles
Created `exploitation_analytics`.`fact_vehicle_emission`             5,988 rows


## 6. Tweet Feature Extraction

Tweets are different from the other structured datasets because their main analytical value is inside free text. This section converts tweet text into structured features and separates multi-valued hashtags into a bridge table.

| Table | Grain | Purpose |
|---|---|---|
| `fact_tweet_features` | One row per tweet | Converts tweet text into numerical and categorical features such as word count, hashtag count, URL count, alert-keyword flag, and simple sentiment label. |
| `bridge_tweet_hashtag` | One row per tweet-hashtag pair | Resolves the one-to-many relationship between tweets and hashtags, making hashtag frequency and hashtag-disaster analysis possible. |

In [8]:
# Extract lightweight text features directly in ClickHouse.
# ifNull(tweet_text, '') prevents Nullable(String) values from producing invalid Nullable(Array) results.
recreate_table(
    "fact_tweet_features",
    f"""
    SELECT
        {source_col(TWEET_TABLE, 'id')} AS tweet_id,
        cityHash64({source_col(TWEET_TABLE, 'disaster_type')}) AS disaster_type_key,
        {source_col(TWEET_TABLE, 'disaster_type')} AS disaster_type,
        length({source_col(TWEET_TABLE, 'emojis')}) AS emoji_count,
        ifNull({source_col(TWEET_TABLE, 'tweet_text')}, '') AS tweet_text,
        lengthUTF8(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, '')) AS text_length,
        length(splitByChar(' ', trim(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, '')))) AS word_count,
        length({source_col(TWEET_TABLE, 'hashtags')}) AS hashtag_count,
        countMatches(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), '@[A-Za-z0-9_]+') AS mention_count,
        countMatches(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'https?://|www\\.') AS url_count,
        countMatches(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), '!') AS exclamation_count,
        countMatches(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), '\\?') AS question_count,
        if(positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'youtube') > 0, 1, 0) AS contains_youtube,
        if(
            positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'emergency') > 0
            OR positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'warning') > 0
            OR positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'evacuation') > 0,
            1,
            0
        ) AS contains_alert_keyword,
        (
            if(positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'safe') > 0, 1, 0)
            + if(positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'rescue') > 0, 1, 0)
            + if(positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'help') > 0, 1, 0)
            - if(positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'damage') > 0, 1, 0)
            - if(positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'dead') > 0, 1, 0)
            - if(positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'destroyed') > 0, 1, 0)
        ) AS simple_sentiment_score,
        multiIf(simple_sentiment_score > 0, 'positive', simple_sentiment_score < 0, 'negative', 'neutral') AS simple_sentiment_label
    FROM {trusted_ref(TWEET_TABLE)}
    """,
    "tweet_id",
)


[table build] Creating `exploitation_analytics`.`fact_tweet_features` from sources=natural_disaster_tweets,global_warming_dataset,temperature_change,co2_emission_by_vehicles
Created `exploitation_analytics`.`fact_tweet_features`             107,593 rows


In [9]:
# Split trusted hashtag arrays out so each tweet-hashtag relationship becomes one analyzable row.
recreate_table(
    "bridge_tweet_hashtag",
    f"""
    SELECT
        {source_col(TWEET_TABLE, 'id')} AS tweet_id,
        lowerUTF8(replaceRegexpAll(hashtag, '^#', '')) AS hashtag
    FROM {trusted_ref(TWEET_TABLE)}
    ARRAY JOIN {source_col(TWEET_TABLE, 'hashtags')} AS hashtag
    WHERE hashtag != ''
    """,
    "(hashtag, tweet_id)",
)

# Split trusted emoji arrays out so each tweet-emoji relationship becomes one analyzable row.
recreate_table(
    "bridge_tweet_emoji",
    f"""
    SELECT
        {source_col(TWEET_TABLE, 'id')} AS tweet_id,
        emoji AS emoji
    FROM {trusted_ref(TWEET_TABLE)}
    ARRAY JOIN {source_col(TWEET_TABLE, 'emojis')} AS emoji
    WHERE emoji != ''
    """,
    "(emoji, tweet_id)",
)


[table build] Creating `exploitation_analytics`.`bridge_tweet_hashtag` from sources=natural_disaster_tweets,global_warming_dataset,temperature_change,co2_emission_by_vehicles
Created `exploitation_analytics`.`bridge_tweet_hashtag`            117,531 rows
[table build] Creating `exploitation_analytics`.`bridge_tweet_emoji` from sources=natural_disaster_tweets,global_warming_dataset,temperature_change,co2_emission_by_vehicles
Created `exploitation_analytics`.`bridge_tweet_emoji`                2,725 rows


## 7. Denormalized Analytical Marts

Marts are query-friendly tables designed for dashboards, reports, and direct analysis. They intentionally denormalize dimensions and facts so users do not need to write many joins.

| Table | Grain | Purpose |
|---|---|---|
| `mart_climate_country_year` | One country per year | Dashboard-ready climate mart with derived metrics such as CO2 per capita, GDP per capita, and emission intensity. |
| `mart_temperature_trends` | One area per month per year | Time-series mart with annual average temperature change and rolling five-year monthly averages. |
| `mart_vehicle_emission_summary` | One make/model/class/fuel group | Summarizes average CO2 emissions and fuel consumption, with ranking inside each vehicle class. |
| `mart_disaster_tweet_features` | One row per tweet | ML/BI-ready tweet feature table without raw modelling joins. |
| `mart_disaster_type_profile` | One row per disaster type | Aggregated profile comparing tweet volume, text length, hashtag usage, alert-keyword share, and simple sentiment by disaster category. |

In [10]:
# Create dashboard-ready marts for climate, temperature, and vehicle-emission analysis.
# These marts join dimension labels back onto facts and add derived analytical metrics.
recreate_table(
    "mart_climate_country_year",
    f"""
    SELECT
        c.country_name AS country,
        y.year,
        y.decade,
        f.temperature_anomaly,
        f.average_temperature,
        f.co2_emissions,
        f.population,
        if(f.population = 0, NULL, f.co2_emissions / f.population) AS co2_per_capita,
        f.gdp,
        if(f.population = 0, NULL, f.gdp / f.population) AS gdp_per_capita,
        if(f.gdp = 0, NULL, f.co2_emissions / f.gdp) AS emission_intensity,
        f.renewable_energy_usage,
        f.forest_area,
        f.deforestation_rate,
        f.extreme_weather_events,
        f.policy_score,
        f.air_pollution_index,
        f.biodiversity_index,
        f.fossil_fuel_usage
    FROM {exploitation_ref('fact_climate_country_year')} f
    INNER JOIN {exploitation_ref('dim_country')} c USING country_key
    INNER JOIN {exploitation_ref('dim_year')} y USING year_key
    """,
    "(country, year)",
)

# Create tweet-focused marts: one feature-level table for ML and one aggregated profile table for BI.
recreate_table(
    "mart_temperature_trends",
    f"""
    SELECT
        a.area_name AS area,
        y.year,
        p.period_name AS period,
        p.period_type AS period_type,
        f.temperature_change_value,
        avg(f.temperature_change_value) OVER (PARTITION BY a.area_name, y.year) AS annual_avg_temperature_change,
        avg(f.temperature_change_value) OVER (
            PARTITION BY a.area_name, p.period_name
            ORDER BY y.year
            ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
        ) AS rolling_5y_period_avg
    FROM {exploitation_ref('fact_temperature_area_period')} f
    INNER JOIN {exploitation_ref('dim_area')} a USING area_key
    INNER JOIN {exploitation_ref('dim_year')} y USING year_key
    INNER JOIN {exploitation_ref('dim_time_period')} p USING period_key
    """,
    "(area, year, period)",
)

recreate_table(
    "mart_vehicle_emission_summary",
    f"""
    SELECT
        v.make,
        v.model,
        v.vehicle_class,
        v.fuel_type,
        count() AS vehicle_record_count,
        avg(f.co2_emissions_g_km) AS avg_co2_emissions_g_km,
        avg(f.fuel_consumption_comb_l_100_km) AS avg_fuel_consumption_comb_l_100_km,
        avg(f.fuel_consumption_comb_mpg) AS avg_fuel_consumption_comb_mpg,
        rank() OVER (PARTITION BY v.vehicle_class ORDER BY avg(f.co2_emissions_g_km) DESC) AS emission_rank_in_class
    FROM {exploitation_ref('fact_vehicle_emission')} f
    INNER JOIN {exploitation_ref('dim_vehicle')} v USING vehicle_key
    GROUP BY v.make, v.model, v.vehicle_class, v.fuel_type
    """,
    "(vehicle_class, emission_rank_in_class, make, model)",
)


[table build] Creating `exploitation_analytics`.`mart_climate_country_year` from sources=natural_disaster_tweets,global_warming_dataset,temperature_change,co2_emission_by_vehicles
Created `exploitation_analytics`.`mart_climate_country_year`       100,000 rows
[table build] Creating `exploitation_analytics`.`mart_temperature_trends` from sources=natural_disaster_tweets,global_warming_dataset,temperature_change,co2_emission_by_vehicles
Created `exploitation_analytics`.`mart_temperature_trends`         241,893 rows
[table build] Creating `exploitation_analytics`.`mart_vehicle_emission_summary` from sources=natural_disaster_tweets,global_warming_dataset,temperature_change,co2_emission_by_vehicles
Created `exploitation_analytics`.`mart_vehicle_emission_summary`      1,856 rows


In [11]:
# Create tweet-focused marts: one feature-level table for ML and one aggregated profile table for BI.
recreate_table(
    "mart_disaster_tweet_features",
    f"""
    SELECT
        tweet_id,
        disaster_type,
        emoji_count,
        text_length,
        word_count,
        hashtag_count,
        mention_count,
        url_count,
        exclamation_count,
        question_count,
        contains_youtube,
        contains_alert_keyword,
        simple_sentiment_score,
        simple_sentiment_label
    FROM {exploitation_ref('fact_tweet_features')}
    """,
    "tweet_id",
)


recreate_table(
    "mart_disaster_type_profile",
    f"""
    SELECT
        disaster_type,
        count() AS tweet_count,
        avg(text_length) AS avg_text_length,
        avg(word_count) AS avg_word_count,
        avg(hashtag_count) AS avg_hashtag_count,
        avg(mention_count) AS avg_mention_count,
        avg(url_count) AS avg_url_count,
        avg(emoji_count) AS avg_emoji_count,
        avg(contains_alert_keyword) AS alert_keyword_share,
        avg(simple_sentiment_score) AS avg_simple_sentiment_score,
        countIf(simple_sentiment_label = 'positive') AS positive_tweets,
        countIf(simple_sentiment_label = 'negative') AS negative_tweets,
        countIf(simple_sentiment_label = 'neutral') AS neutral_tweets
    FROM {exploitation_ref('fact_tweet_features')}
    GROUP BY disaster_type
    """,
    "disaster_type",
)


[table build] Creating `exploitation_analytics`.`mart_disaster_tweet_features` from sources=natural_disaster_tweets,global_warming_dataset,temperature_change,co2_emission_by_vehicles
Created `exploitation_analytics`.`mart_disaster_tweet_features`    107,593 rows
[table build] Creating `exploitation_analytics`.`mart_disaster_type_profile` from sources=natural_disaster_tweets,global_warming_dataset,temperature_change,co2_emission_by_vehicles
Created `exploitation_analytics`.`mart_disaster_type_profile`            5 rows


## 8. Validation and Preview

In [12]:
# List every exploitation table and validate that each output contains the expected number of rows.
exploitation_tables = client.query(
    f"""
    SELECT name
    FROM system.tables
    WHERE database = '{EXPLOITATION_DB}'
    ORDER BY name
    """
).result_rows

for batch_no, (table_name,) in enumerate(exploitation_tables, start=1):
    row_count = client.query(f"SELECT count() FROM {exploitation_ref(table_name)}").first_row[0]
    print(f"[exploitation validation batch {batch_no}/{len(exploitation_tables)}] {table_name:<40} {row_count:>10,} rows")


[exploitation validation batch 1/17] bridge_tweet_emoji                            2,725 rows
[exploitation validation batch 2/17] bridge_tweet_hashtag                        117,531 rows
[exploitation validation batch 3/17] dim_area                                        247 rows
[exploitation validation batch 4/17] dim_country                                     195 rows
[exploitation validation batch 5/17] dim_disaster_type                                 5 rows
[exploitation validation batch 6/17] dim_time_period                                  17 rows
[exploitation validation batch 7/17] dim_vehicle                                   3,097 rows
[exploitation validation batch 8/17] dim_year                                        124 rows
[exploitation validation batch 9/17] fact_climate_country_year                   100,000 rows
[exploitation validation batch 10/17] fact_temperature_area_period                241,893 rows
[exploitation validation batch 11/17] fact_tweet_features  

In [13]:
# Preview the main marts so the notebook output can be inspected immediately after execution.
preview_queries = {
    "mart_climate_country_year": f"SELECT * FROM {exploitation_ref('mart_climate_country_year')} LIMIT 5",
    "mart_temperature_trends": f"SELECT * FROM {exploitation_ref('mart_temperature_trends')} LIMIT 5",
    "mart_vehicle_emission_summary": f"SELECT * FROM {exploitation_ref('mart_vehicle_emission_summary')} LIMIT 5",
    "mart_disaster_type_profile": f"SELECT * FROM {exploitation_ref('mart_disaster_type_profile')} ORDER BY tweet_count DESC LIMIT 10",
}

for batch_no, (title, query) in enumerate(preview_queries.items(), start=1):
    print(f"\n[preview batch {batch_no}/{len(preview_queries)}] {title}")
    display(client.query_df(query))



[preview batch 1/4] mart_climate_country_year


,country,year,decade,temperature_anomaly,average_temperature,co2_emissions,population,co2_per_capita,gdp,gdp_per_capita,...,extreme_weather_events,policy_score,air_pollution_index,biodiversity_index,fossil_fuel_usage,source_system,source_assets,created_at,schema_version,exploitation_asset_type
0,country_1,1900,1900,0.772039,13.235857,2.111143e+08,5.576762e+08,0.378561,3.466274e+12,6215.566881,...,10,60.284283,150.584727,79.475557,57.699551,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart
1,country_1,1900,1900,0.868484,1.096845,7.737906e+08,5.443298e+08,1.421547,1.941820e+12,3567.359459,...,15,10.122462,283.977691,34.448477,44.694559,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart
2,country_1,1900,1900,-1.522098,16.997096,6.083792e+08,2.020119e+08,3.011601,3.694880e+12,18290.409641,...,41,58.360755,147.254164,13.971719,11.643323,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart
3,country_1,1900,1900,-1.458531,-9.319760,5.735942e+05,1.961684e+08,0.002924,9.190033e+12,46847.676235,...,41,43.001773,61.708875,55.262046,95.672073,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart
4,country_1,1901,1900,1.043901,7.210238,9.882580e+08,5.083236e+08,1.944151,6.917356e+11,1360.817525,...,24,16.003608,88.623903,94.277594,96.483974,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart



[preview batch 2/4] mart_temperature_trends


,area,year,period,period_type,temperature_change_value,annual_avg_temperature_change,rolling_5y_period_avg,source_system,source_assets,created_at,schema_version,exploitation_asset_type
0,afghanistan,1961,april,month,-1.786,-0.020294,-1.786,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart
1,afghanistan,1961,august,month,0.361,-0.020294,0.361,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart
2,afghanistan,1961,dec-jan-feb,season,-0.763,-0.020294,-0.763,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart
3,afghanistan,1961,december,month,0.546,-0.020294,0.546,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart
4,afghanistan,1961,february,month,-1.787,-0.020294,-1.787,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart



[preview batch 3/4] mart_vehicle_emission_summary


,make,model,vehicle_class,fuel_type,vehicle_record_count,avg_co2_emissions_g_km,avg_fuel_consumption_comb_l_100_km,avg_fuel_consumption_comb_mpg,emission_rank_in_class,source_system,source_assets,created_at,schema_version,exploitation_asset_type
0,rolls-royce,phantom drophead coupe,compact,z,4,398.500000,17.150000,16.5,1,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart
1,rolls-royce,phantom coupe,compact,z,4,398.000000,17.150000,16.5,2,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart
2,rolls-royce,dawn,compact,z,3,395.666667,16.933333,17.0,3,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart
3,bentley,continental gt convertible,compact,z,1,389.000000,16.600000,17.0,4,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart
4,bentley,continental supersports,compact,z,1,389.000000,16.600000,17.0,4,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart



[preview batch 4/4] mart_disaster_type_profile


,disaster_type,tweet_count,avg_text_length,avg_word_count,avg_hashtag_count,avg_mention_count,avg_url_count,avg_emoji_count,alert_keyword_share,avg_simple_sentiment_score,positive_tweets,negative_tweets,neutral_tweets,source_system,source_assets,created_at,schema_version,exploitation_asset_type
0,hurricane,52084,123.278876,19.422049,0.984448,0.550764,0.073305,0.039436,0.080831,0.051206,9013,6750,36321,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart
1,earthquake,19045,107.135836,15.824731,1.112365,0.501917,0.345655,0.010396,0.033762,0.119821,3521,1342,14182,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart
2,flood,19013,153.905223,23.026719,1.379319,0.756535,0.259033,0.007521,0.032451,0.205544,4482,974,13557,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart
3,wildfire,12996,144.879040,22.826716,0.886042,0.620037,0.087104,0.018082,0.055863,-0.006387,1968,2020,9008,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart
4,cyclone,4455,186.772391,28.446914,1.645791,0.969921,0.089113,0.021324,0.053872,0.096745,900,468,3087,trusted-zone,"natural_disaster_tweets,global_warming_dataset...",2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1,mart


## 9. Exploitation Governance Outputs

<!-- CODEX_EXPLOITATION_GOVERNANCE_STRUCTURED -->

The cells below create a minimal exploitation catalogue, quality summary, and lineage table. They mirror the production DAG governance outputs without changing the business tables.

### Governance Helper Functions


In [14]:
# CODEX_EXPLOITATION_GOVERNANCE_STRUCTURED
# Build data-product catalogue, quality summary, and lineage tables for structured exploitation outputs.

def governance_domain(asset_name: str) -> str:
    if "tweet" in asset_name or "disaster" in asset_name:
        return "disaster_response"
    if "climate" in asset_name or "temperature" in asset_name or asset_name in {"dim_year", "dim_country", "dim_area", "dim_time_period"}:
        return "climate_environment"
    if "vehicle" in asset_name:
        return "vehicle_emissions"
    return "shared_analytics"


def governance_downstream_usage(asset_name: str) -> str:
    if asset_name.startswith("mart_"):
        return "BI dashboard and analyst SQL"
    if asset_name == "fact_tweet_features":
        return "natural disaster classifier"
    if asset_name.startswith("dim_") or asset_name.startswith("bridge_"):
        return "analytical modelling support"
    return "analyst SQL"


def governance_quality_record(asset_name, check_name, status, observed_value, expected_value):
    return {
        "asset_name": asset_name,
        "check_name": check_name,
        "status": status,
        "observed_value": str(observed_value),
        "expected_value": expected_value,
        "created_at": EXPLOITATION_CREATED_AT,
        "schema_version": EXPLOITATION_SCHEMA_VERSION,
    }


def dimension_key_column(asset_name: str):
    return {
        "dim_year": "year_key",
        "dim_country": "country_key",
        "dim_area": "area_key",
        "dim_time_period": "period_key",
        "dim_disaster_type": "disaster_type_key",
        "dim_vehicle": "vehicle_key",
    }.get(asset_name)

business_tables = [
    row[0]
    for row in client.query(f"""
        SELECT name
        FROM system.tables
        WHERE database = '{EXPLOITATION_DB}'
          AND name NOT IN ('exploitation_catalogue', 'exploitation_quality_summary', 'exploitation_lineage')
        ORDER BY name
    """).result_rows
]



### Build Governance Records


In [15]:
catalogue_records = []
quality_records = []
lineage_records = []
metadata_columns = ["source_system", "source_assets", "created_at", "schema_version", "exploitation_asset_type"]
for batch_no, asset_name in enumerate(business_tables, start=1):
    ref = exploitation_ref(asset_name)
    row_count = int(client.query(f"SELECT count() FROM {ref}").first_row[0])
    print(f"[governance scan batch {batch_no}/{len(business_tables)}] {asset_name}: rows={row_count:,}")
    source_assets = client.query(f"SELECT any(source_assets) FROM {ref}").first_row[0] if row_count else ""
    quality_records.append(governance_quality_record(asset_name, "non_empty", "PASS" if row_count > 0 else "FAIL", row_count, "> 0"))
    missing_metadata = int(client.query(f"""
        SELECT count()
        FROM {ref}
        WHERE {" OR ".join(f"{quote_identifier(column)} IS NULL OR toString({quote_identifier(column)}) = ''" for column in metadata_columns)}
    """).first_row[0])
    quality_records.append(governance_quality_record(asset_name, "metadata_complete", "PASS" if missing_metadata == 0 else "FAIL", missing_metadata, "0 missing rows"))
    key_column = dimension_key_column(asset_name)
    if key_column:
        distinct_count = int(client.query(f"SELECT uniqExact({quote_identifier(key_column)}) FROM {ref}").first_row[0])
        quality_records.append(governance_quality_record(asset_name, "dimension_key_unique", "PASS" if distinct_count == row_count else "FAIL", f"{distinct_count}/{row_count}", "distinct=count"))
    catalogue_records.append({
        "asset_name": asset_name,
        "domain": governance_domain(asset_name),
        "asset_type": exploitation_asset_type(asset_name),
        "storage_system": "ClickHouse",
        "location": f"{EXPLOITATION_DB}.{asset_name}",
        "source_system": "trusted-zone",
        "source_assets": source_assets,
        "downstream_usage": governance_downstream_usage(asset_name),
        "owner": "analytics",
        "record_count": row_count,
        "created_at": EXPLOITATION_CREATED_AT,
        "schema_version": EXPLOITATION_SCHEMA_VERSION,
    })
    lineage_records.append({
        "asset_name": asset_name,
        "upstream_zone": "trusted-zone",
        "upstream_assets": source_assets,
        "transformation_name": f"structured_exploitation.{asset_name}",
        "downstream_usage": governance_downstream_usage(asset_name),
        "created_at": EXPLOITATION_CREATED_AT,
        "schema_version": EXPLOITATION_SCHEMA_VERSION,
    })

quality_by_asset = {}
for record in quality_records:
    if record["status"] != "PASS":
        quality_by_asset[record["asset_name"]] = record["status"]
for record in catalogue_records:
    record["quality_status"] = quality_by_asset.get(record["asset_name"], "PASS")



[governance scan batch 1/17] bridge_tweet_emoji: rows=2,725
[governance scan batch 2/17] bridge_tweet_hashtag: rows=117,531
[governance scan batch 3/17] dim_area: rows=247
[governance scan batch 4/17] dim_country: rows=195
[governance scan batch 5/17] dim_disaster_type: rows=5
[governance scan batch 6/17] dim_time_period: rows=17
[governance scan batch 7/17] dim_vehicle: rows=3,097
[governance scan batch 8/17] dim_year: rows=124
[governance scan batch 9/17] fact_climate_country_year: rows=100,000
[governance scan batch 10/17] fact_temperature_area_period: rows=241,893
[governance scan batch 11/17] fact_tweet_features: rows=107,593
[governance scan batch 12/17] fact_vehicle_emission: rows=5,988
[governance scan batch 13/17] mart_climate_country_year: rows=100,000
[governance scan batch 14/17] mart_disaster_tweet_features: rows=107,593
[governance scan batch 15/17] mart_disaster_type_profile: rows=5
[governance scan batch 16/17] mart_temperature_trends: rows=241,893
[governance scan batc

### Create Governance Tables


In [16]:
client.command(f"DROP TABLE IF EXISTS {exploitation_ref('exploitation_catalogue')}")
client.command(f"""
CREATE TABLE {exploitation_ref('exploitation_catalogue')}
(
    asset_name String, domain String, asset_type String, storage_system String, location String,
    source_system String, source_assets String, downstream_usage String, owner String,
    quality_status String, record_count UInt64, created_at String, schema_version String
)
ENGINE = MergeTree()
ORDER BY (domain, asset_name)
""")
client.command(f"DROP TABLE IF EXISTS {exploitation_ref('exploitation_quality_summary')}")
client.command(f"""
CREATE TABLE {exploitation_ref('exploitation_quality_summary')}
(
    asset_name String, check_name String, status String, observed_value String,
    expected_value String, created_at String, schema_version String
)
ENGINE = MergeTree()
ORDER BY (asset_name, check_name)
""")
client.command(f"DROP TABLE IF EXISTS {exploitation_ref('exploitation_lineage')}")
client.command(f"""
CREATE TABLE {exploitation_ref('exploitation_lineage')}
(
    asset_name String, upstream_zone String, upstream_assets String, transformation_name String,
    downstream_usage String, created_at String, schema_version String
)
ENGINE = MergeTree()
ORDER BY (asset_name, upstream_zone)
""")



### Write Governance Outputs and Preview


In [17]:
client.insert(
    f"{EXPLOITATION_DB}.exploitation_catalogue",
    [[record[column] for column in ["asset_name", "domain", "asset_type", "storage_system", "location", "source_system", "source_assets", "downstream_usage", "owner", "quality_status", "record_count", "created_at", "schema_version"]] for record in catalogue_records],
    column_names=["asset_name", "domain", "asset_type", "storage_system", "location", "source_system", "source_assets", "downstream_usage", "owner", "quality_status", "record_count", "created_at", "schema_version"],
)
client.insert(
    f"{EXPLOITATION_DB}.exploitation_quality_summary",
    [[record[column] for column in ["asset_name", "check_name", "status", "observed_value", "expected_value", "created_at", "schema_version"]] for record in quality_records],
    column_names=["asset_name", "check_name", "status", "observed_value", "expected_value", "created_at", "schema_version"],
)
client.insert(
    f"{EXPLOITATION_DB}.exploitation_lineage",
    [[record[column] for column in ["asset_name", "upstream_zone", "upstream_assets", "transformation_name", "downstream_usage", "created_at", "schema_version"]] for record in lineage_records],
    column_names=["asset_name", "upstream_zone", "upstream_assets", "transformation_name", "downstream_usage", "created_at", "schema_version"],
)

print(f"Wrote exploitation_catalogue rows={len(catalogue_records)}")
print(f"Wrote exploitation_quality_summary rows={len(quality_records)}")
print(f"Wrote exploitation_lineage rows={len(lineage_records)}")
display(client.query_df(f"SELECT * FROM {exploitation_ref('exploitation_quality_summary')} ORDER BY asset_name, check_name"))


Wrote exploitation_catalogue rows=17
Wrote exploitation_quality_summary rows=40
Wrote exploitation_lineage rows=17


,asset_name,check_name,status,observed_value,expected_value,created_at,schema_version
0,bridge_tweet_emoji,metadata_complete,PASS,0,0 missing rows,2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1
1,bridge_tweet_emoji,non_empty,PASS,2725,> 0,2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1
2,bridge_tweet_hashtag,metadata_complete,PASS,0,0 missing rows,2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1
3,bridge_tweet_hashtag,non_empty,PASS,117531,> 0,2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1
4,dim_area,dimension_key_unique,PASS,247/247,distinct=count,2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1
5,dim_area,metadata_complete,PASS,0,0 missing rows,2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1
6,dim_area,non_empty,PASS,247,> 0,2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1
7,dim_country,dimension_key_unique,PASS,195/195,distinct=count,2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1
8,dim_country,metadata_complete,PASS,0,0 missing rows,2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1
9,dim_country,non_empty,PASS,195,> 0,2026-06-06T17:09:19.161423+00:00,exploitation_structured_v1
